# View intensity statistics

Visualises the per-FOV intensity statistics produced by the FOV scheduler (nb05).

For each **bit** (color channel in each hybridisation round) two plots are generated:

1. **Z-profile** — median pixel intensity vs z position; one line per FOV (same color, α = 0.5).
2. **FOV heatmap** — spatial map of median intensity; FOV stage positions are converted to a regular (x_idx, y_idx) grid.

The bead channel (488 nm) and bead-plane frames (z = 0) are excluded automatically.

## 1 — Setup

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.colors as mcolors

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/after_imaging/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.progress        import ProgressTracker
from MERci.analysis.view_intensity_stats import load_stats_with_annotations
from MERci.plots.view_intensity_stats_plots import plot_z_profiles, plot_fov_heatmap

print(f"SAMPLE_DIR : {SAMPLE_DIR}")

## 2 — Experiment parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
IMAGE_SUFFIX = ".zarr"          # must match what was used in nb05

# ── Plotting parameters ────────────────────────────────────────────────
# Which rounds to show (None = all rounds with completed data)
SHOW_ROUNDS  = None

# Which color channels to show in nm (None = all non-bead channels)
# e.g. [560, 650, 750] to show only the MERFISH bits channels
SHOW_COLORS  = None

# Z value used for bead/fiducial frames — these are excluded from plots
BEAD_Z       = 0.0

# Wavelength of the fiducial bead channel — excluded from plots
BEAD_COLOR   = 488.0

print(f"Sample name  : {SAMPLE_NAME}")
print(f"Positions tag: {POSITIONS_TAG}")
print(f"Image suffix : {IMAGE_SUFFIX}")

In [ ]:
config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = SAMPLE_DIR / "metadata",
    analysis_dir   = SAMPLE_DIR / "analysis",
    settings_dir   = SAMPLE_DIR / "settings",
    round_info_csv = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
)

meta    = ExperimentMetadata.load(config.round_info_csv, config.positions_txt,
                                   config.data_dir, image_suffix=config.image_suffix)
tracker = ProgressTracker(config.analysis_dir)

print(f"Rounds : {meta.n_rounds}")
print(f"FOVs   : {meta.n_fovs}")

## 3 — Load stats data

Reads all completed stats CSVs and merges them with frame-table information (color, z position).
Results are stored in `stats_df` — a single DataFrame with one row per (FOV, frame).

In [ ]:
stats_df = load_stats_with_annotations(config, meta, tracker)

if stats_df.empty:
    print("No stats data found. Run nb05 first to generate stats CSVs.")
else:
    n_fovs_loaded = stats_df.groupby(["round_id", "fov_id"]).ngroups
    print(f"Loaded stats for {n_fovs_loaded} FOV×round combinations.")
    if "color" in stats_df.columns:
        colors = sorted(stats_df["color"].dropna().unique().astype(int))
        print(f"Colors found   : {colors}")

## 4 — Z-profiles

For each bit (round × color channel), plot median pixel intensity vs z position.  
Each line represents one FOV; lines are overlapping with α = 0.5.

In [ ]:
if stats_df.empty:
    print("No data to plot.")
elif "color" not in stats_df.columns:
    print("Column 'color' missing — frame table was not found for this experiment.")
else:
    rounds = sorted(stats_df["round_id"].unique())
    if SHOW_ROUNDS is not None:
        rounds = [r for r in rounds if r in SHOW_ROUNDS]

    for rid in rounds:
        rdata   = stats_df[
            (stats_df["round_id"] == rid)
            & (stats_df["z"] != BEAD_Z)
        ]
        colors  = sorted(rdata["color"].dropna().unique())
        colors  = [c for c in colors if round(c) != round(BEAD_COLOR)]
        if SHOW_COLORS is not None:
            colors = [c for c in colors if int(round(c)) in SHOW_COLORS]

        for c in colors:
            plot_z_profiles(stats_df, rid, c, BEAD_Z, BEAD_COLOR)

## 5 — FOV intensity heatmaps

For each bit, the median intensity of each FOV (median across z positions) is placed on a 2-D grid
derived from the stage positions.  
Brighter = higher signal.

In [ ]:
if stats_df.empty:
    print("No data to plot.")
elif "color" not in stats_df.columns:
    print("Column 'color' missing — frame table was not found for this experiment.")
else:
    rounds = sorted(stats_df["round_id"].unique())
    if SHOW_ROUNDS is not None:
        rounds = [r for r in rounds if r in SHOW_ROUNDS]

    for rid in rounds:
        rdata  = stats_df[
            (stats_df["round_id"] == rid)
            & (stats_df["z"] != BEAD_Z)
        ]
        colors = sorted(rdata["color"].dropna().unique())
        colors = [c for c in colors if round(c) != round(BEAD_COLOR)]
        if SHOW_COLORS is not None:
            colors = [c for c in colors if int(round(c)) in SHOW_COLORS]

        for c in colors:
            plot_fov_heatmap(stats_df, meta, rid, c, BEAD_Z, BEAD_COLOR)